# 03. Statistical Inference & Hypothesis Testing

## Objective
Formally test the statistical significance, direction, and effect size of relationships between daily social media habits and mental wellbeing indicators (`N = 481` survey respondents).

## Research Hypotheses ($\alpha = 0.05$)
* **H1 (Bivariate Association):** Daily social media screen time and compulsive usage habits correlate positively with `mental_distress_score` and `sleep_issues_score`.
* **H2 (Tier Differences):** Median `mental_distress_score` differs significantly across `Low (<2h)`, `Moderate (2-5h)`, and `High (>5h)` usage tiers.
* **H3 (Multivariate Controls):** Specific behavioral habits (`purposeless_use`, `restlessness`, `social_comparison`) remain significant predictors of mental distress even after controlling for `daily_screen_time`, `age`, and `gender`.

### Step 1: Load Cleaned Survey Data
Import statistical libraries (`scipy.stats`, `statsmodels`) and load `cleaned_wellbeing_data.csv`.


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load processed dataset from Notebook 1
DATA_PATH = '../data/processed/cleaned_wellbeing_data.csv'
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df)} records for hypothesis testing.")


Loaded 481 records for hypothesis testing.


### Step 2: Shapiro-Wilk Normality Tests
Formally test the null hypothesis of normality ($p > 0.05$) across all target outcomes and predictors to validate non-parametric test selection.


In [2]:
vars_to_test = [
    'daily_screen_time',
    'mental_distress_score',
    'depression_score',
    'sleep_issues_score',
    'worries_score',
    'platform_count'
]

normality_results = []
for col in vars_to_test:
    stat, p_val = stats.shapiro(df[col].dropna())
    normality_results.append({
        'Variable': col,
        'W-Statistic': round(stat, 4),
        'p-value': f"{p_val:.4e}",
        'Normal (p > 0.05)': 'Yes' if p_val > 0.05 else 'No (Use Non-Parametric)'
    })

display(pd.DataFrame(normality_results))


,Variable,W-Statistic,p-value,Normal (p > 0.05)
0,daily_screen_time,0.9069,1.4147e-16,No (Use Non-Parametric)
1,mental_distress_score,0.9736,1.2722e-07,No (Use Non-Parametric)
2,depression_score,0.8985,2.4607e-17,No (Use Non-Parametric)
3,sleep_issues_score,0.8690,1.1217e-19,No (Use Non-Parametric)
4,worries_score,0.8733,2.3280e-19,No (Use Non-Parametric)
5,platform_count,0.9585,2.1569e-10,No (Use Non-Parametric)


### Step 3. Bivariate Correlation Analysis (H1)
Because Likert-scale survey scores and binned screen time are ordinal/non-normal, we compute **Spearman’s rank correlation ($\rho$)** alongside **Pearson’s $r$** to measure how strongly daily screen time and specific usage behaviors associate with mental distress and sleep issues.


In [3]:
predictors = [
    'daily_screen_time',
    'platform_count',
    'purposeless_use',
    'distracted_busy',
    'restlessness',
    'social_comparison',
    'validation_seeking'
]
outcomes = ['mental_distress_score', 'depression_score', 'sleep_issues_score']

corr_rows = []
for outcome in outcomes:
    for pred in predictors:
        sub = df[[pred, outcome]].dropna()
        rho, p_spearman = stats.spearmanr(sub[pred], sub[outcome])
        r, p_pearson = stats.pearsonr(sub[pred], sub[outcome])
        corr_rows.append({
            'Outcome': outcome,
            'Predictor': pred,
            'Spearman_rho': round(rho, 3),
            'Spearman_p': f"{p_spearman:.4e}",
            'Significant (alpha=0.05)': 'Yes' if p_spearman < 0.05 else 'No'
        })

corr_df = pd.DataFrame(corr_rows)
display(corr_df.sort_values(by=['Outcome', 'Spearman_rho'], ascending=[True, False]))


,Outcome,Predictor,Spearman_rho,Spearman_p,Significant (alpha=0.05)
12,depression_score,social_comparison,0.411,4.5265e-21,Yes
10,depression_score,distracted_busy,0.365,1.4299e-16,Yes
11,depression_score,restlessness,0.323,3.8480e-13,Yes
7,depression_score,daily_screen_time,0.318,8.4235e-13,Yes
9,depression_score,purposeless_use,0.295,3.9132e-11,Yes
13,depression_score,validation_seeking,0.266,3.0314e-09,Yes
8,depression_score,platform_count,0.199,1.0652e-05,Yes
3,mental_distress_score,distracted_busy,0.483,1.6369e-29,Yes
5,mental_distress_score,social_comparison,0.468,1.4225e-27,Yes
4,mental_distress_score,restlessness,0.438,6.3393e-24,Yes


### Step 4. Group Comparison Across Usage Tiers (H2)
We compare `mental_distress_score` across `Low (<2h)`, `Moderate (2-5h)`, and `High (>5h)` usage groups using the non-parametric **Kruskal-Wallis H-test** (alongside One-Way ANOVA for comparison). If significant, we run **post-hoc Mann-Whitney U tests with Bonferroni correction** to identify which specific tiers differ.


In [4]:
tiers = ['Low (<2h)', 'Moderate (2-5h)', 'High (>5h)']
groups = [df[df['usage_tier'] == t]['mental_distress_score'].dropna() for t in tiers]

# 1. Omnibus Test: Kruskal-Wallis & One-Way ANOVA
kw_stat, kw_p = stats.kruskal(*groups)
f_stat, anova_p = stats.f_oneway(*groups)

# 2. Effect Size (Epsilon-squared for Kruskal-Wallis)
n = sum(len(g) for g in groups)
eps_sq = kw_stat / ((n**2 - 1) / (n + 1))

print(f"Kruskal-Wallis Test: H = {kw_stat:.4f}, p = {kw_p:.4e}, Effect Size (epsilon^2) = {eps_sq:.4f}")
print(f"One-Way ANOVA (Ref): F = {f_stat:.4f}, p = {anova_p:.4e}\n")

# 3. Post-Hoc Pairwise Mann-Whitney U Tests (with Bonferroni correction for 3 comparisons)
if kw_p < 0.05:
    print("--- Post-Hoc Pairwise Comparisons (Bonferroni-adjusted alpha = 0.0167) ---")
    pairs = list(combinations(tiers, 2))
    bonferroni_alpha = 0.05 / len(pairs)
    
    posthoc_rows = []
    for t1, t2 in pairs:
        g1 = df[df['usage_tier'] == t1]['mental_distress_score'].dropna()
        g2 = df[df['usage_tier'] == t2]['mental_distress_score'].dropna()
        u_stat, p_val = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        posthoc_rows.append({
            'Comparison': f"{t1} vs. {t2}",
            'Median_1': g1.median(),
            'Median_2': g2.median(),
            'U-Statistic': u_stat,
            'p-value': f"{p_val:.4e}",
            'Significant (Bonferroni)': 'Yes' if p_val < bonferroni_alpha else 'No'
        })
    display(pd.DataFrame(posthoc_rows))


Kruskal-Wallis Test: H = 62.9997, p = 2.0883e-14, Effect Size (epsilon^2) = 0.1312
One-Way ANOVA (Ref): F = 39.2647, p = 1.6279e-16

--- Post-Hoc Pairwise Comparisons (Bonferroni-adjusted alpha = 0.0167) ---


,Comparison,Median_1,Median_2,U-Statistic,p-value,Significant (Bonferroni)
0,Low (<2h) vs. Moderate (2-5h),13.0,18.0,7873.0,3.5158e-10,Yes
1,Low (<2h) vs. High (>5h),13.0,19.0,2517.5,8.1828e-14,Yes
2,Moderate (2-5h) vs. High (>5h),18.0,19.0,12108.0,1.8753e-03,Yes


### Step 5. Multiple Linear Regression (Controlling for Confounders — H3)
Finally, we fit an Ordinary Least Squares (OLS) regression model to determine whether daily screen time and specific digital habits predict `mental_distress_score` when controlling for respondent `age` and `gender`.


In [5]:
# Fit OLS model controlling for age and gender
formula = (
    "mental_distress_score ~ daily_screen_time + purposeless_use + "
    "restlessness + social_comparison + platform_count + age + C(gender)"
)

model = smf.ols(formula=formula, data=df).fit(cov_type='HC3')  # HC3 robust standard errors
print(model.summary())


                              OLS Regression Results                             
Dep. Variable:     mental_distress_score   R-squared:                       0.450
Model:                               OLS   Adj. R-squared:                  0.438
Method:                    Least Squares   F-statistic:                     39.99
Date:                   Fri, 27 Nov 2026   Prob (F-statistic):           8.99e-57
Time:                           02:01:11   Log-Likelihood:                -1308.1
No. Observations:                    481   AIC:                             2638.
Df Residuals:                        470   BIC:                             2684.
Df Model:                             10                                         
Covariance Type:                     HC3                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------

### Step 6: Statistical Conclusions
* **H1 Supported:** Both `daily_screen_time` and behavioral habit scores show statistically significant positive correlations ($p < 0.001$) with mental distress and sleep disruption.
* **H2 Supported:** Mental distress differs significantly across usage tiers, with the largest jump occurring between **Low (<2h)** (Median = 13.0) and **Moderate/High (≥2h)** users (Medians = 18.0 and 19.0).
* **H3 Supported:** Behavioral engagement patterns (`social_comparison`, `restlessness`, `purposeless_use`) explain substantial variance in distress beyond raw screen duration alone.
